# 02 — Extract Pose Keypoints from the Full UCF101 Dataset

**Goal:** save one `.npz` file per video listed in `data/processed/manifest.csv`.

This version processes **the full manifest** (`MAX_VIDEOS = None`), resumes by skipping valid keypoint files that already exist, uses collision-resistant filenames, and periodically saves the extraction report. It does not train the LSTM; it prepares pose data for notebooks 03–05.

The extractor currently selects the largest detected person in each sampled frame. Multi-person tracking is not implemented in this baseline.


In [6]:
from pathlib import Path
import sys, json, time, hashlib, re
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
PROJECT_DIR = Path(
    r"C:\Users\admin\ViTPose_Action_Recognition"
)

sys.path.insert(0, str(PROJECT_DIR))
from src.vitpose_model import (
    ViTPose, get_affine_transform, decode_heatmaps,
    IMG_H, IMG_W, IMG_MEAN, IMG_STD
)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ckpt_path = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
model = ViTPose().to(DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()
detector = YOLO("yolo11n.pt")

manifest_path = PROJECT_DIR / "data" / "processed" / "manifest.csv"
manifest = pd.read_csv(manifest_path)
assert len(manifest), "Manifest is empty. Run notebook 00 first."

KEYPOINT_DIR = PROJECT_DIR / "data" / "processed" / "keypoints"
KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)

FRAME_STEP = 2       # process every 2nd frame; set 1 for every frame
MAX_VIDEOS = None    # process the FULL manifest
REPORT_EVERY = 25    # write progress report after every 25 manifest rows

required_cols = {"video_path", "label"}
missing_cols = required_cols - set(manifest.columns)
if missing_cols:
    raise ValueError(f"Manifest is missing required columns: {sorted(missing_cols)}")

print(f"Manifest videos: {len(manifest):,}")
print(f"Unique action classes: {manifest['label'].nunique():,}")
print(f"Device: {DEVICE}")
print(f"Keypoint output directory: {KEYPOINT_DIR.resolve()}")


Manifest videos: 13,320
Unique action classes: 101
Device: cuda:0
Keypoint output directory: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints


In [7]:
def pose_for_box(frame_rgb, bbox):
    x1, y1, x2, y2 = map(float, bbox)
    bw, bh = x2-x1, y2-y1
    center = np.array([(x1+x2)/2, (y1+y2)/2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    trans = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(frame_rgb, trans, (IMG_W, IMG_H))
    crop = crop.astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    x = torch.from_numpy(crop.transpose(2,0,1).copy()).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        heatmaps = model(x)[0].float().cpu().numpy()
    return decode_heatmaps(heatmaps, center, scale)

def extract_video(video_path, label, frame_step=2):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, {"status": "open_failed", "frames": 0}
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frame_indices, keypoint_rows, boxes_rows = [], [], []
    frame_i = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        if frame_i % frame_step == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            det = detector.predict(frame_bgr, classes=[0], conf=0.30, verbose=False)[0]
            if len(det.boxes):
                boxes = det.boxes.xyxy.cpu().numpy()
                areas = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
                bbox = boxes[np.argmax(areas)]
                kps = pose_for_box(frame_rgb, bbox)
            else:
                bbox = np.full(4, np.nan, dtype=np.float32)
                kps = np.full((17,3), np.nan, dtype=np.float32)
            frame_indices.append(frame_i)
            keypoint_rows.append(kps.astype(np.float32))
            boxes_rows.append(np.asarray(bbox, dtype=np.float32))
        frame_i += 1
    cap.release()
    if not frame_indices:
        return None, {"status": "no_frames", "frames": 0}
    return {
        "keypoints": np.stack(keypoint_rows),  # (T, 17, 3)
        "boxes": np.stack(boxes_rows),         # (T, 4)
        "frame_indices": np.asarray(frame_indices, dtype=np.int32),
        "label": np.asarray(str(label)),
        "video_path": np.asarray(str(video_path)),
        "fps": np.asarray(float(fps)),
        "frame_step": np.asarray(int(frame_step)),
    }, {"status": "ok", "frames": frame_i, "sampled_frames": len(frame_indices)}

In [ ]:
# Full-dataset extraction with resume support.
# Existing valid .npz files are kept; only missing/invalid videos are processed.

def output_path_for(video_path, label):
    # Reuse an older pilot filename if it belongs to this exact video.
    legacy_path = KEYPOINT_DIR / f"{video_path.stem}.npz"
    if legacy_path.exists():
        try:
            with np.load(legacy_path, allow_pickle=False) as old:
                old_video = str(old["video_path"].item()) if "video_path" in old.files else ""
            if old_video and Path(old_video).resolve() == video_path.resolve():
                return legacy_path
        except Exception:
            pass

    # Include label and a path hash to avoid overwriting videos with duplicate stems.
    safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", str(label)).strip("_") or "unknown"
    path_hash = hashlib.sha1(str(video_path.resolve()).encode("utf-8")).hexdigest()[:10]
    return KEYPOINT_DIR / f"{safe_label}__{video_path.stem}__{path_hash}.npz"


def valid_existing_file(path, expected_video_path):
    try:
        with np.load(path, allow_pickle=False) as saved:
            needed = {"keypoints", "boxes", "frame_indices", "label", "video_path"}
            if not needed.issubset(saved.files):
                return False, 0
            kps = saved["keypoints"]
            boxes = saved["boxes"]
            frames = saved["frame_indices"]
            saved_path = str(saved["video_path"].item())
            if kps.ndim != 3 or kps.shape[1:] != (17, 3):
                return False, 0
            if boxes.shape != (len(kps), 4) or frames.shape != (len(kps),):
                return False, 0
            if Path(saved_path).resolve() != expected_video_path.resolve():
                return False, 0
            return True, int(len(kps))
    except Exception:
        return False, 0


records = manifest if MAX_VIDEOS is None else manifest.head(MAX_VIDEOS)
report_path = PROJECT_DIR / "data" / "processed" / "extraction_report.csv"
rows = []
total = len(records)

for position, (_, row) in enumerate(records.iterrows(), start=1):
    video_path = Path(str(row["video_path"]))
    label = str(row["label"])
    out_path = output_path_for(video_path, label)
    print(f"[{position}/{total}] {label}: {video_path.name}")

    started = time.time()
    try:
        is_valid, saved_frames = valid_existing_file(out_path, video_path)
        if is_valid:
            status = {
                "status": "ok",
                "skipped_existing": True,
                "frames": None,
                "sampled_frames": saved_frames,
                "output_path": str(out_path.resolve()),
            }
            print(f"  Already extracted; keeping {saved_frames} sampled frames.")
        else:
            data, status = extract_video(video_path, label, FRAME_STEP)
            if data is not None:
                # Write to a temporary .npz then replace the final file atomically.
                temp_path = out_path.with_name(out_path.stem + ".tmp.npz")
                np.savez_compressed(temp_path, **data)
                temp_path.replace(out_path)
                status["output_path"] = str(out_path.resolve())
            else:
                status["output_path"] = ""
            status["skipped_existing"] = False

        status.update({
            "video_path": str(video_path),
            "label": label,
            "seconds": round(time.time() - started, 1),
        })
    except Exception as e:
        status = {
            "status": "error",
            "skipped_existing": False,
            "video_path": str(video_path),
            "label": label,
            "output_path": str(out_path.resolve()) if out_path.exists() else "",
            "seconds": round(time.time() - started, 1),
            "error": repr(e),
        }
        print("  ERROR:", repr(e))

    rows.append(status)

    # Periodically persist progress so a long run is recoverable.
    if position % REPORT_EVERY == 0:
        pd.DataFrame(rows).to_csv(report_path, index=False)
        ok_count = sum(r.get("status") == "ok" for r in rows)
        error_count = sum(r.get("status") == "error" for r in rows)
        print(f"Progress saved: {position}/{total}; successful/existing={ok_count}; errors={error_count}")

pd.DataFrame(rows).to_csv(report_path, index=False)
report_df = pd.DataFrame(rows)
print("\nExtraction finished.")
print("Manifest rows processed:", len(report_df))
print("Successful (including existing):", int((report_df["status"] == "ok").sum()))
print("Errors:", int((report_df["status"] == "error").sum()))
print("Report saved to:", report_path.resolve())


[1/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g01_c01.avi
  Already extracted; keeping 82 sampled frames.
[2/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g01_c02.avi
  Already extracted; keeping 62 sampled frames.
[3/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g01_c03.avi
  Already extracted; keeping 130 sampled frames.
[4/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g01_c04.avi
[5/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g01_c05.avi
[6/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g01_c06.avi
[7/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g02_c01.avi
[8/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g02_c02.avi
[9/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g02_c03.avi
[10/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g02_c04.avi
[11/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g03_c01.avi
[12/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g03_c02.avi
[13/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g03_c03.avi
[14/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g03_c04.avi
[15/13320] ApplyEyeMakeup: v_ApplyEyeMakeup_g03_c05.avi
[16/13320] ApplyEyeMakeu

In [10]:
# Inspect a saved keypoint file and summarize extraction coverage.
saved_files = sorted(KEYPOINT_DIR.glob("*.npz"))
print("Saved keypoint files:", len(saved_files))

if saved_files:
    with np.load(saved_files[0], allow_pickle=False) as sample:
        for key in sample.files:
            arr = sample[key]
            print(key, arr.shape if hasattr(arr, "shape") else type(arr),
                  arr.dtype if hasattr(arr, "dtype") else "")
        print("Example keypoints array shape:", sample["keypoints"].shape)

if report_path.exists():
    report = pd.read_csv(report_path)
    print("\nReport rows:", len(report))
    if "status" in report.columns:
        print("Statuses:\n", report["status"].value_counts(dropna=False).to_string())
    if "label" in report.columns:
        print("Classes represented in report:", report["label"].nunique())


Saved keypoint files: 13320
keypoints (118, 17, 3) float32
boxes (118, 4) float32
frame_indices (118,) int32
label () <U14
video_path () <U105
fps () float64
frame_step () int64
Example keypoints array shape: (118, 17, 3)

Report rows: 13320
Statuses:
 status
ok    13320
Classes represented in report: 101
